# Volatility Regime Detection - Exploratory Data Analysis

This notebook demonstrates the mathematical prototyping of the volatility state transition logic before it was moved to the production `utils.py` module.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# Append parent directory to path so we can import our engine modules
sys.path.append(os.path.abspath('..'))
from utils import load_sample_data, detect_regimes

plt.style.use('dark_background')

### 1. Generate Geometric Brownian Motion Data

In [ ]:
df = load_sample_data(num_days=500, initial_price=100.0)
df.head()

### 2. Prototype Volatility Regime Detection
We use a rolling 20-day standard deviation, annualized by $\sqrt{252}$.

In [ ]:
daily_returns = np.log(df['Close'] / df['Close'].shift(1))
rolling_vol = daily_returns.rolling(window=20).std() * np.sqrt(252)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(df['Close'], color='cyan', label='Price')
ax1.set_ylabel('Asset Price ($)')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(rolling_vol, color='magenta', alpha=0.5, label='Rolling 20D Vol')
ax2.axhline(0.15, color='green', linestyle='--', alpha=0.5, label='Low Vol Threshold')
ax2.axhline(0.25, color='red', linestyle='--', alpha=0.5, label='High Vol Threshold')
ax2.set_ylabel('Annualized Volatility')
ax2.legend(loc='lower right')
plt.title('GBM Asset Price vs Underlying Volatility')
plt.show()

### Conclusion
The 20-day annualized threshold successfully captures regime transitions. This logic was extracted into `detect_regimes()` and decoupled into `utils.py` for the production backtesting pipeline.